In [5]:
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from datasets import Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import TrainingArguments, Trainer
import numpy as np
from transformers import DataCollatorWithPadding


In [6]:
# Set the path to the file you'd like to load
fake_path = "Data/Fake.csv"
real_path = "Data/True.csv"
tweets_df_train_path = "Data/train.csv"

# Load datasets
fake_df = pd.read_csv(fake_path)
fake_df["label"] = 1   # fake = 1

real_df = pd.read_csv(real_path)
real_df["label"] = 0   # real = 0

# Combine title and text
real_df["all_text"] = real_df["title"] + " " + real_df["text"]
fake_df["all_text"] = fake_df["title"] + " " + fake_df["text"]

# Load tweet dataset
tweets_df_train = pd.read_csv(tweets_df_train_path)
tweets_df_train = tweets_df_train.rename(columns={"target": "label"})
tweets_df_train = tweets_df_train.rename(columns={"text": "all_text"})

# Combine all datasets
df = pd.concat([
    fake_df[["all_text", "label"]],
    real_df[["all_text", "label"]],
    tweets_df_train[["all_text", "label"]]
], ignore_index=True)

# Split data
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["all_text"], df["label"], test_size=0.2, random_state=42
)

# Convert to Hugging Face datasets
train_dataset = Dataset.from_dict({
    "text": train_texts.tolist(),
    "label": train_labels.tolist()
})

test_dataset = Dataset.from_dict({
    "text": test_texts.tolist(),
    "label": test_labels.tolist()
})

In [ ]:
# Load tokenizer
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenization function (original)
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Remove unused text column
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

# Set format for PyTorch
train_dataset.set_format("torch")
test_dataset.set_format("torch")

In [7]:
#SMALL DATASET

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

# Set format for PyTorch
train_dataset.set_format("torch")
test_dataset.set_format("torch")

Map: 100%|██████████| 10503/10503 [00:03<00:00, 3487.19 examples/s]


In [ ]:
#BIG DATASET

# Load DistilBERT model
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# Metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

# Training settings (original style)
training_args = TrainingArguments(
    output_dir="./distilbert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    load_best_model_at_end=True
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [8]:
#SMALL DATASET

# Dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Load model
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

# Faster training settings
training_args = TrainingArguments(
    output_dir="./distilbert_results",
    eval_strategy="no",
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    report_to="none"
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7508.06it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
# Train model
trainer.train()

/Users/jackfogerty/Documents/Coding/FakeNewsDetection/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
# Evaluate
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)

accuracy = accuracy_score(test_labels, y_pred)
print(f"DistilBERT Accuracy: {accuracy:.2%}")

print("\nClassification Report:")
print(classification_report(test_labels, y_pred, target_names=["real", "fake"]))